# Function 8

In [1]:

import numpy as np
import matplotlib.pyplot as plt
import math
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import qmc



### Data preparation

Here we prepare the initial data provided

In [2]:
input = np.load('./initial_inputs.npy')
output = np.load('./initial_outputs.npy')
print(input)
print(output)
#print(input.shape)
#print(input.shape[1])
func_dimensions = input.shape[1]
print('this function has ', func_dimensions, ' dimensions')

[[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.41010452
  

Here we add the data provided with the weekly queries

In [3]:
additionalInputs = [[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5], [0.061949, 0.086118, 0.024059, 0.022119, 0.358532, 0.798558, 0.447126, 0.928641], [0.061949, 0.086118, 0.024059, 0.022119, 0.358532, 0.798558, 0.447126, 0.928641], [0.061949, 0.086118, 0.024059, 0.022119, 0.358532, 0.798558, 0.447126, 0.928641], [0.061949, 0.086118, 0.024059, 0.022119, 0.358532, 0.798558, 0.447126, 0.928641], [0.0, 0.25, 0.0, 0.0, 0.0, 0.125, 0.0, 1.0], [0.0, 0.5, 0.0, 0.25, 0.5, 0.125, 0.0, 0.0], [0.0, 0.25, 0.0, 1.0, 0.25, 0.75, 0.25, 1.0], [0.233201, 0.617495, 0.446795, 0.493155, 0.788961, 0.61884, 0.307125, 0.23723], [0.0, 0.0, 0.25, 0.0, 0.75, 0.875, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.875, 0.0, 1.0], [0.807272, 0.850723, 0.389804, 0.323567, 0.686034, 0.322757, 0.579367, 0.063267]]
additionalOutputs = [np.float64(8.798300000000001), np.float64(9.6234724089539), np.float64(9.6234724089539), np.float64(9.6234724089539), np.float64(9.6234724089539), np.float64(9.340175), np.float64(9.495175), np.float64(8.96205), np.float64(9.2768352281725), np.float64(9.633925), np.float64(9.627675), np.float64(7.9213439762321)]

input = np.append(input, additionalInputs, axis=0)
output = np.append(output, additionalOutputs)

print(input)
print(output)

[[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.41010452
  

# Bayesian Optimisation approach
We approach the study of this function with the Bayesian Optimisation
using and adaptation of the UCB acquisition function from required assignment 12.1

# Preparation of the exploration space
Here we prepare the exploration space of the function<br>
The values are increments of 0.01 bounded between 0 and 1 (included) => 101 values for each dimension of the space<br>
dimension of the current function is stored in <b>func_dimensions</b>

In [4]:
#Initialise query lists and maximum observations
X, Y = input, output

# --- Latin Hypercube Sampling (LHS) ---
n_samples = 9200000
sampler_lhs = qmc.LatinHypercube(d=func_dimensions, seed=42)
x_grid = sampler_lhs.random(n=n_samples)
#print(f"LHS shape: {x_grid_lhs.shape}")

#x_grid = np.delete(x_grid, 0, axis=0)
#print(x_grid)
print(x_grid.shape)


(9200000, 8)


In [5]:
print(x_grid[:10])

[[0.96471024 0.50747506 0.0143748  0.83000427 0.45804597 0.56175783
  0.9782134  0.64740404]
 [0.9639689  0.5703518  0.8494117  0.8176749  0.77386971 0.53074393
  0.31200136 0.47248704]
 [0.57536603 0.04774706 0.34315871 0.3510083  0.40034079 0.65905474
  0.00193555 0.58280762]
 [0.18392383 0.79670357 0.2685893  0.46079282 0.441579   0.04770406
  0.28973992 0.76992946]
 [0.33537257 0.31030887 0.62428984 0.76307802 0.44358368 0.45885864
  0.86075954 0.88521438]
 [0.37262463 0.61701045 0.92070362 0.6716754  0.01229545 0.11088785
  0.28555137 0.42042095]
 [0.51995199 0.24875031 0.90587596 0.74999728 0.44120426 0.41416851
  0.97661568 0.93056948]
 [0.05975669 0.34354744 0.62331498 0.39520694 0.7499096  0.80654973
  0.24844211 0.38061557]
 [0.7251921  0.57529081 0.1372044  0.85810888 0.02998945 0.21677191
  0.97265074 0.30433833]
 [0.70206132 0.08670671 0.16322217 0.43820834 0.42243845 0.35488123
  0.33741385 0.82122568]]


# Bayesian Optimisation with UCB applied

In [6]:
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
real_noise_std = 1e-10
noise_assumption = 1e-10

#Define kernel of GP
kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')

model = GaussianProcessRegressor(kernel = kernel)
#Fit the model
model.fit(np.array(X), np.array(Y).reshape(-1, 1))


#Calculate the mean and standard deviation and make them one-dimensional for plotting
post_mean, post_std = model.predict(x_grid, return_std=True)

#Acquisition function parameter
#beta = 1.96
#beta = 0.5
beta = 0.02

acquisition_function = post_mean + beta * post_std

grid = x_grid.squeeze()
obs = grid[np.argmax(acquisition_function)] #Else use the acquisition function

print('Next observation: ', obs)
#print (obs)


Next observation:  [0.37178627 0.21726819 0.50013187 0.66441657 0.1148562  0.29984719
 0.51035    0.5844518 ]
